# Qwen3 Layer Export with Quantization

This notebook:
1. Loads Qwen3-4B with **INT4 quantization** (bitsandbytes)
2. Wraps model to capture hidden states after each layer
3. Saves complete model directory to Google Drive

**Output:** Full quantized model (~2.4 GB) with layer capture capability

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/EVA_Ai_Exports/qwen_layer_model_4bit"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Install dependencies
!pip install -q transformers bitsandbytes accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, BitsAndBytesConfig
import gc
import os

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Model configuration
MODEL_ID = "RefalMachine/RuadaptQwen3-4B-Instruct"
NUM_LAYERS = 36

# INT4 Quantization config (4-bit, NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("INT4 Quantization config:")
print(f"  load_in_4bit: {bnb_config.load_in_4bit}")
print(f"  bnb_4bit_quant_type: {bnb_config.bnb_4bit_quant_type}")

In [ ]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded")

In [ ]:
# Load quantized model
print(f"Loading model with INT4 quantization from {MODEL_ID}...")
print("This may take 10-20 minutes...")

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()
print("Model loaded!")

# Memory info
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

In [ ]:
# Test inference
print("Testing inference...")
test_text = "Привет! Как дела?"
inputs = tokenizer(test_text, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)
    
print(f"Logits shape: {outputs.logits.shape}")
print(f"Number of hidden states: {len(outputs.hidden_states)}")
for i, hs in enumerate(outputs.hidden_states[:3]):
    print(f"  Hidden state {i}: {hs.shape}")

In [ ]:
# Layer capture wrapper
class LayerCaptureWrapper(torch.nn.Module):
    """Wrapper that captures hidden states after each layer"""
    
    def __init__(self, base_model, num_layers):
        super().__init__()
        self.base_model = base_model
        self.num_layers = num_layers
        self.layer_outputs = {}
        
        # Register hooks on each layer
        for idx, layer in enumerate(base_model.model.layers):
            layer.register_forward_hook(self._hook_fn(idx))
    
    def _hook_fn(self, layer_idx):
        def hook(module, input, output):
            self.layer_outputs[layer_idx] = output[0].detach()
        return hook
    
    def forward(self, input_ids, attention_mask=None, **kwargs):
        self.layer_outputs.clear()
        return self.base_model(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
    
    def get_layer_outputs(self, input_ids, attention_mask=None):
        """Return dict of layer_idx -> hidden_state tensor"""
        self.layer_outputs.clear()
        
        with torch.no_grad():
            outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        
        result = {}
        for idx in range(self.num_layers):
            if idx in self.layer_outputs:
                result[idx] = self.layer_outputs[idx]
        
        return result, outputs.logits

# Create wrapper
wrapper = LayerCaptureWrapper(model, NUM_LAYERS)
print(f"LayerCaptureWrapper created with {NUM_LAYERS} layers")

In [ ]:
# Test layer capture
print("Testing layer capture...")
test_text = "Привет!"
inputs = tokenizer(test_text, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

layer_outputs, logits = wrapper.get_layer_outputs(**inputs)

print(f"Captured {len(layer_outputs)} layer outputs")
for idx in [0, 10, 20, 35]:
    if idx in layer_outputs:
        print(f"  Layer {idx}: {layer_outputs[idx].shape}")

In [ ]:
# Save complete model to Google Drive
print(f"Saving model to {OUTPUT_DIR}...")

# Save quantized model (includes model.safetensors + config)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved!")

# Check saved files
!ls -lh $OUTPUT_DIR
!du -sh $OUTPUT_DIR

In [ ]:
# Save checkpoint with layer info for EVA
print("Saving layer capture checkpoint...")

checkpoint = {
    'layer_info': {
        'num_layers': NUM_LAYERS,
        'hidden_size': config.hidden_size,
        'is_quantized': True,
        'quantization_type': 'int4'
    },
    'config': config,
    'tokenizer_path': OUTPUT_DIR  # tokenizer is saved separately
}

torch.save(checkpoint, f"{OUTPUT_DIR}/layer_capture_info.pt")

print(f"Layer info saved to {OUTPUT_DIR}/layer_capture_info.pt")

In [ ]:
# Verify we can load the saved model
print("Testing reload from saved model...")

del model
gc.collect()
torch.cuda.empty_cache()

# Reload model
reloaded_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    quantization_config=bnb_config,
    device_map="auto"
)
reloaded_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

# Test inference
test_text = "Привет!"
inputs = reloaded_tokenizer(test_text, return_tensors="pt")
inputs = {k: v.to(reloaded_model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = reloaded_model(**inputs)

print(f"Reload test passed! Logits shape: {outputs.logits.shape}")

In [ ]:
# Summary
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Model: {MODEL_ID}")
print(f"Quantization: INT4 (NF4)")
print(f"Layers: {NUM_LAYERS}")
print(f"Hidden size: {config.hidden_size}")
print(f"Output dir: {OUTPUT_DIR}")
print("\nFiles saved:")
!ls -lh $OUTPUT_DIR
print("=" * 60)